[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/18_tag_3_5_dateien_streamen_tfdata.ipynb)

# Tag 3-5 - 18 Daten aus einzelnen Dateien laden, ohne alles in den RAM zu legen

Viele Datensätze liegen nicht als eine große Matrix im Speicher vor, sondern als einzelne Dateien:

- Bilder in Ordnern,
- Audiodateien,
- Textdateien,
- viele kleine Messdateien.

Dann ist es meistens falsch, erst alles mit Python in eine riesige Liste zu laden. Besser ist eine Streaming-Pipeline mit `tf.data`:

1. Dateipfade sammeln,
2. Datei pro Sample lesen,
3. dekodieren und vorverarbeiten,
4. batches bilden,
5. parallelisieren und prefetching nutzen.


## Lesepfad für dieses Notebook

Dieses Notebook trennt bewusst drei Dinge:

1. Dateien erzeugen: `data_root` ist der Ordner, in dem PNG-Dateien liegen.
2. Pfade sammeln: `image_path_strings` ist nur eine Liste von Dateinamen, noch keine Bilddaten im RAM.
3. Dateien laden: `load_image(path)` liest genau eine Datei, dekodiert sie und gibt `(image, label)` zurück.

Später wird `load_image` in `.map(load_image)` verwendet. Erst dort werden aus Pfaden echte Bildtensoren.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## Mini-Bilddatensatz auf Platte erzeugen

Für ein komplett ausführbares Notebook erzeugen wir kleine PNG-Dateien selbst. Sie werden unter `notebooks/Day_3_5/generated_data/18_dateien_streamen_tfdata/` abgelegt. Dieser Ordner ist in `.gitignore` ausgeschlossen. Die Klassen liegen in Ordnern:

```text
root/
  class_0/
    image_000.png
  class_1/
    image_000.png
```

Das entspricht dem üblichen Layout vieler Bilddatensätze.


In [ ]:
# Die Demo-Dateien werden bewusst im Repository-Ordner erzeugt,
# damit man die Dateistruktur nach dem Ausführen sehen kann.
# Der Ordner ist in .gitignore eingetragen und wird nicht committed.
data_root = Path("notebooks/Day_3_5/generated_data/18_dateien_streamen_tfdata")
if not data_root.parent.exists() and not Path("notebooks/Day_3_5").exists():
    data_root = Path("generated_data/18_dateien_streamen_tfdata")
data_root.mkdir(parents=True, exist_ok=True)
image_size = 32
num_per_class = 180
rng = np.random.default_rng(RANDOM_STATE)

for class_id in [0, 1]:
    class_dir = data_root / f"class_{class_id}"
    class_dir.mkdir(parents=True, exist_ok=True)

    for idx in range(num_per_class):
        image = rng.normal(loc=0.15, scale=0.08, size=(image_size, image_size, 3))
        image = np.clip(image, 0.0, 1.0)

        if class_id == 0:
            image[8:24, 8:24, 0] += 0.65
        else:
            rr, cc = np.ogrid[:image_size, :image_size]
            mask = (rr - 16) ** 2 + (cc - 16) ** 2 <= 9 ** 2
            image[mask, 1] += 0.65

        image = np.clip(image, 0.0, 1.0)
        image_uint8 = (image * 255).astype("uint8")
        png = tf.io.encode_png(image_uint8).numpy()
        (class_dir / f"image_{idx:03d}.png").write_bytes(png)

print(data_root)
print("Dateien:", len(list(data_root.glob("*/*.png"))))


## Kurz erklärt: Warum `.map`, `.shuffle`, `.batch`?

`path_ds`, `train_ds` und `val_ds` sind `tf.data.Dataset`-Objekte. Sie sind also keine normalen Listen und keine selbst geschriebenen Funktionen.

TensorFlow-Datasets haben eingebaute Methoden:

- `.shuffle(...)`: Reihenfolge mischen.
- `.map(load_image)`: auf jedes Element die Funktion `load_image` anwenden.
- `.batch(32)`: immer 32 Elemente zu einem Batch zusammenfassen.
- `.prefetch(...)`: den nächsten Batch vorbereiten.

In diesem Notebook enthält `path_ds` zunächst nur Dateipfade. Erst `.map(load_image)` liest die Dateien wirklich von der Festplatte.


## Dateipfade als Dataset

Statt `tf.data.Dataset.list_files(...)` verwenden wir hier bewusst `pathlib`. Das ist unter Windows robuster, weil TensorFlow-Glob-Patterns mit relativen Backslash-Pfaden manchmal keine Dateien finden.

Wichtig: Die Bilder selbst werden dabei noch nicht geladen. Im Dataset stehen zunächst nur Dateipfade. Geladen wird später in `load_image(...)` pro Sample.


In [ ]:
image_paths = sorted(data_root.glob("*/*.png"))
if not image_paths:
    raise FileNotFoundError(
        f"Keine PNG-Dateien unter {data_root} gefunden. "
        "Bitte die vorherige Zelle zum Erzeugen des Mini-Bilddatensatzes ausführen."
    )

# TensorFlow bekommt Strings. pathlib findet die Dateien zuverlässig auf Windows, macOS und Linux.
image_path_strings = [str(path) for path in image_paths]
# from_tensor_slices macht aus der Python-Liste ein tf.data.Dataset-Objekt.
# Ab jetzt können wir Dataset-Methoden wie .shuffle, .map und .batch verwenden.
path_ds = tf.data.Dataset.from_tensor_slices(image_path_strings)
path_ds = path_ds.shuffle(buffer_size=len(image_path_strings), seed=RANDOM_STATE)

for path in path_ds.take(5):
    print(path.numpy().decode("utf-8"))


## Datei lesen, dekodieren und Label ableiten

Das Label steckt im Namen des Elternordners. `class_0` wird zu `0`, `class_1` zu `1`.

Die Funktion wird später per `map(...)` auf jedes Element angewendet. TensorFlow kann mehrere Dateien parallel lesen und vorbereiten.


In [ ]:
def get_label_from_path(path):
    # path ist ein TensorFlow-String, z. B. ".../class_0/image_000.png".
    # Aus dem Elternordner class_0 oder class_1 leiten wir das Label ab.
    normalized = tf.strings.regex_replace(path, r"\\", "/")
    parts = tf.strings.split(normalized, sep="/")
    class_name = parts[-2]
    class_number = tf.strings.to_number(
        tf.strings.regex_replace(class_name, "class_", ""),
        out_type=tf.int32,
    )
    return tf.cast(class_number, tf.float32)


def load_image(path):
    # Diese Funktion wird später in dataset.map(load_image) auf jeden Pfad angewendet.
    # Input: Dateipfad. Output: Bildtensor und Label.
    label = get_label_from_path(path)
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_png(image_bytes, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, (image_size, image_size))
    return image, label


image_ds = path_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

# Wieder nur zur Kontrolle: Einen Batch holen, um Form und Labels zu sehen.
# Das lädt nicht den ganzen Datensatz in den RAM.
images, labels = next(iter(image_ds.batch(8)))
images.shape, labels.numpy()


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, image, label in zip(axes.ravel(), images, labels):
    ax.imshow(image)
    ax.set_title(f"label={int(label.numpy())}")
    ax.axis("off")
plt.tight_layout()


## Train/Validation-Split als Pipeline

Für große Datensätze würde man Splits oft über feste Dateilisten speichern. Hier nehmen wir für die Demo `take` und `skip`.


In [ ]:
num_files = len(image_path_strings)
num_train = int(num_files * 0.8)

# Split-Idee für diese Demo:
# 1. all_paths enthält nur Dateipfade.
# 2. take(num_train) nimmt die ersten 80 Prozent als Training.
# 3. skip(num_train) nimmt den Rest als Validation.
# Für echte Projekte wären feste Split-Dateien noch besser.
all_paths = tf.data.Dataset.from_tensor_slices(image_path_strings)
all_paths = all_paths.shuffle(buffer_size=num_files, seed=RANDOM_STATE)

train_ds = (
    all_paths.take(num_train)
    # Erst hier werden die Dateien wirklich gelesen und dekodiert.
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(512, seed=RANDOM_STATE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    all_paths.skip(num_train)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

# Kontroll-Batch: 32 Bilder mit Form (32, 32, 32, 3) und 32 Labels.
for batch_images, batch_labels in train_ds.take(1):
    print(batch_images.shape, batch_labels.shape)


## Kleines Modell direkt auf der Datei-Pipeline trainieren

Das Modell sieht nur Batches. Es muss nicht wissen, ob die Bilder aus RAM, einzelnen Dateien oder TFRecords kommen.


In [ ]:
# model.fit bekommt gleich train_ds und val_ds.
# Diese Datasets liefern bereits fertige Batches: (batch_images, batch_labels).
# Das Modell muss deshalb nicht wissen, ob die Bilder vorher aus Dateien gelesen wurden.
model = keras.Sequential(
    [
        keras.Input(shape=(image_size, image_size, 3)),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="file_streaming_cnn",
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.003),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy(name="accuracy")],
)

history = model.fit(train_ds, validation_data=val_ds, epochs=8, verbose=0)
pd.DataFrame(history.history)


## Was wäre die RAM-Variante?

Die RAM-Variante liest alle Bilder sofort in ein Array. Das ist bei 360 kleinen Bildern kein Problem, aber bei echten Datensätzen schnell unmöglich.


In [ ]:
# Für die Größenabschätzung lesen wir bewusst nur eine Datei aus path_ds.
# next(iter(path_ds)) liefert einen einzelnen Dateipfad, nicht den ganzen Datensatz.
one_example_path = next(iter(path_ds))
one_image_bytes = tf.io.read_file(one_example_path).numpy()

# PNG ist komprimiert auf Platte. Ein float32-Tensor im RAM braucht deutlich mehr Bytes.
approx_file_mb = len(one_image_bytes) * num_files / 1024**2
approx_array_mb = num_files * image_size * image_size * 3 * 4 / 1024**2

pd.DataFrame(
    [
        ("komprimierte PNG-Dateien grob", approx_file_mb),
        ("float32 Tensor im RAM grob", approx_array_mb),
    ],
    columns=["variante", "megabyte"],
)


## `cache()` bewusst einsetzen

`cache()` kann eine Pipeline stark beschleunigen, weil Daten nach dem ersten Lesen zwischengespeichert werden. Aber:

- `cache()` ohne Dateipfad cached im RAM,
- bei großen Datensätzen kann das den Speicher füllen,
- bei Datenaugmentation sollte man genau überlegen, ob vor oder nach der Augmentation gecached wird.


In [ ]:
# cache() ist ebenfalls eine Dataset-Methode.
# Hier wird nach dem ersten Lesen zwischengespeichert.
# Bei kleinen Daten ist das bequem, bei großen Daten kann es zu viel RAM brauchen.
cached_train_ds = (
    all_paths.take(num_train)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(512, seed=RANDOM_STATE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

print(cached_train_ds)


## Limitierungen

- `take`/`skip` nach einem zufälligen Shuffle ist für Produktivsplits nicht stabil genug. Besser sind gespeicherte Split-Dateien oder Metadaten mit festen IDs.
- Viele kleine Dateien können langsam sein, weil Dateisystemzugriffe teuer sind. Dann können TFRecords oder größere Shards schneller sein.
- `tf.data.Dataset.list_files(...)` ist bequem, aber unter Windows mit relativen Glob-Patterns manchmal fehleranfällig. Eine explizite `pathlib`-Liste ist leichter zu debuggen.
- `cache()` kann versehentlich doch alles in den RAM laden. Bei großen Daten nur bewusst einsetzen.
- Labels aus Ordnernamen sind bequem, aber nicht immer robust. In echten Projekten sind Metadatentabellen oft sauberer.
- Die Demo nutzt kleine synthetische Bilder. Echte Bilder brauchen oft Augmentation, Qualitätschecks und ggf. Klassenbalancierung.
- Die erzeugten Dateien liegen lokal unter `notebooks/Day_3_5/generated_data/` und sind absichtlich nicht für Git gedacht.
